# 🐍 3 — OOP: Inheritance, Polymorphism & Abstraction (Part 3)

**Welcome to OOP Part 3!** 🎉 This is the heart of Object-Oriented Programming. Real applications (Swiggy, Amazon, YouTube) have **many classes** that relate to each other. Here we learn how classes connect, share, and reuse code.

Every topic is explained in **simple English**, with **real-life examples**, **ASCII diagrams**, and **line-by-line comments**.

> 📝 **A note about the diagrams:** the original notebook loaded diagram images from Google Colab's temporary storage, so those images no longer exist in the file. I have re-drawn each one as a clean text diagram — they are portable and just as clear.

---

## 📚 What you will learn here

| # | Topic |
|---|-------|
| 1 | Class Relationships & **Aggregation** ("has-a") |
| 2 | **Inheritance** ("is-a") |
| 3 | What gets inherited (constructor, attributes, methods) |
| 4 | **Method Overriding** |
| 5 | The **`super()`** keyword |
| 6 | **Types of Inheritance** (single, multilevel, hierarchical, multiple, hybrid) |
| 7 | The **Diamond Problem** & MRO |
| 8 | **Polymorphism** (overriding, overloading, operator overloading) |
| 9 | **Abstraction** (abstract classes & methods) |

---

# 🅰️ Part 1 — Class Relationships & Aggregation

## What are Class Relationships?

When you build a big application, you don't use just one class — you use **many**. For example, a **Swiggy** app might have a `User` class, a `Restaurant` class, a `Payment` class, a `Delivery` class, and so on. These classes often **talk to each other** — they have relationships.

There are **two** main relationships:
1. **Aggregation** (a "has-a" relationship)
2. **Inheritance** (an "is-a" relationship)

## Aggregation — the "has-a" relationship

**Aggregation** means one class **owns** another class. One is the *owner*, the other is its *property*.

🧠 **Examples:**
- A **Customer** *has an* **Address** → Customer owns Address.
- A **Restaurant** *has a* **Menu** → Restaurant owns Menu.

Why make a separate class? Because something like an *address* is complex — it has a city, pin code, state, street, landmark. So we give it its own `Address` class, and the `Customer` class simply *holds* an Address object.

In [ ]:
# Aggregation: a Customer HAS an Address (Customer owns Address)
class Customer:
    def __init__(self, name, gender, address):
        self.name = name
        self.gender = gender
        self.address = address          # this is an Address OBJECT, not simple text

    def print_address(self):
        # reach into the address object and fetch its attributes
        print(self.address.city, self.address.pin, self.address.state)

class Address:
    def __init__(self, city, pin, state):
        self.city = city
        self.pin = pin
        self.state = state

# First make an Address object...
add1 = Address("gurgaon", 122011, "haryana")
# ...then pass it into the Customer object
cust = Customer("nitish", "male", add1)

cust.print_address()

gurgaon 122011 haryana


**Summary:** a `Customer` needs a name, gender, and address. But an address is complex, so it gets its **own class**. We create the `Address` object first, then hand it to the `Customer` — the customer's constructor receives it and stores it. That is aggregation. ✅

### ❓ Question: can we access a *private* attribute across aggregation?

What if the `city` attribute in `Address` is made **private** (`__city`)? Can the `Customer` class still print it? Let's test — this cell gives an **error on purpose**.

In [ ]:
class Customer:
    def __init__(self, name, gender, address):
        self.name = name
        self.gender = gender
        self.address = address

    def print_address(self):
        print(self.address.__city, self.address.pin, self.address.state)  # trying private

class Address:
    def __init__(self, city, pin, state):
        self.__city = city    # __city is now PRIVATE
        self.pin = pin
        self.state = state

add1 = Address("gurgaon", 122011, "haryana")
cust = Customer("nitish", "male", add1)
cust.print_address()   # ❌ AttributeError: can't access the private __city

**The answer is NO** — you cannot directly access a private attribute of a class you own. The correct way is to add a **getter method** inside `Address` that returns the private value.

In [ ]:
class Customer:
    def __init__(self, name, gender, address):
        self.name = name
        self.gender = gender
        self.address = address

    def print_address(self):
        print(self.address.get_city(), self.address.pin, self.address.state)  # use the getter

class Address:
    def __init__(self, city, pin, state):
        self.__city = city
        self.pin = pin
        self.state = state

    def get_city(self):        # a GETTER: safely returns the private value
        return self.__city

add1 = Address("gurgaon", 122011, "haryana")
cust = Customer("nitish", "male", add1)
cust.print_address()

gurgaon 122011 haryana


### Using methods across aggregation

The owner class can also call methods of the class it owns. Here `edit_profile` in `Customer` updates the name *and* calls `edit_address` inside the `Address` object.

In [ ]:
class Customer:
    def __init__(self, name, gender, address):
        self.name = name
        self.gender = gender
        self.address = address

    def print_address(self):
        print(self.address.get_city(), self.address.pin, self.address.state)

    def edit_profile(self, new_name, new_city, new_pin, new_state):
        self.name = new_name
        self.address.edit_address(new_city, new_pin, new_state)   # call Address's method

class Address:
    def __init__(self, city, pin, state):
        self.__city = city
        self.pin = pin
        self.state = state

    def get_city(self):
        return self.__city

    def edit_address(self, new_city, new_pin, new_state):
        self.__city = new_city
        self.pin = new_pin
        self.state = new_state

add1 = Address("gurgaon", 122011, "haryana")
cust = Customer("nitish", "male", add1)
cust.print_address()

cust.edit_profile("ankit", "mumbai", 1234112, "maharashtra")
cust.print_address()

gurgaon 122011 haryana
mumbai 1234112 maharashtra


### 📐 Aggregation Class Diagram

```
   ┌───────────┐   has-a   ┌───────────┐
   │ Customer  │◆─────────▶│  Address  │
   │───────────│           │───────────│
   │ + name    │           │ - city    │
   │ + gender  │           │ + pin     │
   │ + address │           │ + state   │
   └───────────┘           └───────────┘
```
- The **diamond ◆** sits on the **owner** side (Customer owns Address).
- **`-`** means a **private** attribute; **`+`** means a **public** attribute.

---

# 🅱️ Part 2 — Inheritance

## What is Inheritance?

**Inheritance** is an **"is-a"** relationship. A **child class** inherits from a **parent class**, and the child gets access to the parent's **data and methods**.

🧠 **Simple idea:** whatever belongs to the father also belongs to the son.

### 🎯 The biggest benefit: code reusability

Inheritance follows the **DRY** principle — **Don't Repeat Yourself**. Instead of writing the same code (like `login` / `register`) in every class, you write it **once** in a parent class, and all child classes reuse it.

```
        ┌──────────┐
        │   User   │   ← parent class
        │──────────│
        │ + login()│
        │ + register()
        └────▲─────┘
             │ inherits (is-a)
     ┌───────┴───────┐
┌────┴─────┐   ┌─────┴──────┐
│ Student  │   │ Instructor │   ← child classes
└──────────┘   └────────────┘
```
The arrow head points **to the parent**.

### First, without inheritance (to see the problem)

Here `Student` does **not** inherit from `User`, so it is completely independent — it cannot see `User`'s `name` or `login()`. This errors on purpose.

In [ ]:
# parent class
class User:
    def __init__(self):
        self.name = "nitish"
    def login(self):
        print("login")

# child class — NOT inheriting from User
class Student:
    def enroll(self):
        print("enroll into the course")

u = User()
s = Student()

print(s.name)   # ❌ AttributeError: Student has no attribute 'name'
# s.login()     # would also fail — Student can't see User's methods
# s.enroll()    # this one works (it's defined in Student)

> 📌 If a child class does **not** inherit, it has **no access** to the parent's variables, methods, or constructor.

### Now, with inheritance

To inherit, put the parent's name in **brackets**: `class Student(User)`. Now `Student` can access everything in `User`.

In [ ]:
# parent class
class User:
    def __init__(self):
        self.name = "nitish"
    def login(self):
        print("login")

# child class — inherits from User (note the brackets)
class Student(User):
    def enroll(self):
        print("enroll into the course")

u = User()
s = Student()

print(s.name)   # works — inherited from User
s.login()       # works — inherited method
s.enroll()      # works — its own method

nitish
login
enroll into the course


**Summary:** to make a class inherit, write the parent's name inside brackets in the child's definition: `class Child(Parent)`. Then the child can use the parent's **data, methods, and constructor**.

```
     ┌──────────┐
     │   User   │  parent
     └────▲─────┘
          │  Student(User)
     ┌────┴─────┐
     │ Student  │  child
     └──────────┘
```

## 3️⃣ What Exactly Gets Inherited?

1. The **constructor**
2. **Non-private** attributes (variables)
3. **Non-private** methods

Let's explore each through a `Phone` (parent) and `Smartphone` (child) example.

### Concept 1 — child has NO constructor → parent's constructor runs

If the child has no `__init__`, Python uses the **parent's** constructor automatically.

In [ ]:
# parent class
class Phone:
    def __init__(self, price, brand, camera):
        print("Inside phone constructor")
        self.price = price
        self.brand = brand
        self.camera = camera
    def buy(self):
        print("buying a phone")

# child class with NO constructor of its own
class Smartphone(Phone):
    pass

s = Smartphone(20000, "Apple", 13)   # uses Phone's constructor
s.buy()                              # buy() is a non-private method -> inherited

Inside phone constructor
buying a phone


### Concept 2 — child HAS its own constructor → parent's is NOT called

If the child defines its own `__init__`, the parent's constructor does **not** run automatically. So the parent's variables never get created.

In [ ]:
class Phone:
    def __init__(self, price, brand, camera):
        print("Inside phone constructor")
        self.price = price
        self.brand = brand
        self.camera = camera

class Smartphone(Phone):
    def __init__(self, os, ram):     # child has its OWN constructor
        self.os = os
        self.ram = ram
        print("Inside smartphone constructor")

s = Smartphone("Android", 2)
# s.brand would fail here, because Phone's constructor never ran.
# To fix that, we use super() (coming soon).

Inside smartphone constructor


### Concept 3 — private members are NOT inherited

A child object can access everything **except** the parent's **private** data and **private** methods.

In [ ]:
# private ATTRIBUTE is not accessible in the child
class Phone:
    def __init__(self, price, brand, camera):
        print("Inside phone constructor")
        self.__price = price     # private
        self.brand = brand
        self.camera = camera
    def show(self):
        print(self.__price)

class Smartphone(Phone):
    def check(self):
        print(self.__price)      # trying to use the parent's private -> fails

s = Smartphone(20000, "Apple", 13)
print(s.brand)   # works (public)
s.check()        # ❌ AttributeError: cannot access the private __price

Inside phone constructor
Apple


In [ ]:
# private METHOD is not accessible either
class Phone:
    def __init__(self, price, brand, camera):
        print("Inside phone constructor")
        self.__price = price
        self.brand = brand
        self.camera = camera
    def __show(self):            # private method
        print(self.__price)

class Smartphone(Phone):
    def check(self):
        print(self.__price)

s = Smartphone(20000, "Apple", 13)
s.__show()   # ❌ AttributeError: __show is private, not accessible

Inside phone constructor


> 📝 **Three concepts to remember:**
> 1. Child has **no** constructor → parent's constructor runs.
> 2. Child **has** its own constructor → parent's constructor does **not** run (so parent's variables aren't created).
> 3. Child can access everything **except** the parent's **private** members.

### Getter still works across inheritance

Even though `__num` is private, a **getter** method (`get_num`) can expose it to the child.

In [ ]:
class Parent:
    def __init__(self, num):
        self.__num = num
    def get_num(self):        # getter for the private value
        return self.__num

class Child(Parent):
    def show(self):
        print("This is in child class")

son = Child(100)          # child has no constructor -> Parent's runs
print(son.get_num())      # getter returns the private value
son.show()

100
This is in child class


### When the child has its own constructor, the parent's private breaks

Here `Child` has its own `__init__`, so `Parent.__init__` never runs, and `get_num()` fails.

In [ ]:
class Parent:
    def __init__(self, num):
        self.__num = num
    def get_num(self):
        return self.__num

class Child(Parent):
    def __init__(self, val, num):
        self.__val = val          # only __val is set; Parent's __num is never created
    def get_val(self):
        return self.__val

son = Child(100, 10)
print("Parent Num:", son.get_num())   # ❌ fails: __num was never set
print("Child val:", son.get_val())

### A cleaner flow example

Here `B` inherits from `A` and has no constructor, so `A`'s constructor runs and sets `var1 = 100`. Then `display1` (a parent method) prints it.

In [ ]:
class A:
    def __init__(self):
        self.var1 = 100
    def display1(self, var1):
        print("Class A :", self.var1)

class B(A):
    def display2(self, var1):
        print("Class B:", self.var1)

obj = B()
obj.display1(200)   # B has no constructor -> A's runs -> var1 = 100

Class A : 100


## 4️⃣ Method Overriding

**Method overriding** happens when the **child and parent both have a method with the same name**. In that case, the **child's version wins** when called on a child object.

In [ ]:
class Phone:
    def __init__(self, price, brand, camera):
        print("Inside phone constructor")
        self.__price = price
        self.brand = brand
        self.camera = camera
    def buy(self):
        print("buying a phone")

class Smartphone(Phone):
    def buy(self):                      # same name as parent's method
        print("buying a smartphone")

s = Smartphone(20000, "Apple", 13)
s.buy()   # the CHILD's buy() runs (overrides the parent's)

Inside phone constructor
buying a smartphone


## 5️⃣ The `super()` Keyword ⭐

`super()` lets a child class **call the parent's methods and constructor**. Use it when you want to *add to* the parent's behaviour instead of fully replacing it.

### Using `super()` to call a parent method

The child's `buy()` runs first, then `super().buy()` calls the parent's `buy()` too.

In [ ]:
class Phone:
    def __init__(self, price, brand, camera):
        print("Inside phone constructor")
        self.__price = price
        self.brand = brand
        self.camera = camera
    def buy(self):
        print("buying a phone")

class Smartphone(Phone):
    def buy(self):
        print("buying a Smartphone")
        super().buy()          # also call the parent's buy()

s = Smartphone(20000, "Apple", 13)
s.buy()

Inside phone constructor
buying a Smartphone
buying a phone


### Using `super()` to call the parent's constructor

This is the **most common use**. When the child has its own constructor, call `super().__init__(...)` to also run the parent's constructor — so the parent's variables get created too.

In [ ]:
class Phone:
    def __init__(self, price, brand, camera):
        print("Inside phone constructor")
        self.__price = price
        self.brand = brand
        self.camera = camera

class Smartphone(Phone):
    def __init__(self, price, brand, camera, os, ram):
        print("Inside smartphone constructor")
        super().__init__(price, brand, camera)   # run the parent's constructor
        self.os = os
        self.ram = ram

s = Smartphone(20000, "Samsung", 12, "Android", 2)
print(s.os)      # from the child
print(s.brand)   # from the parent (thanks to super())

Inside smartphone constructor
Inside phone constructor
Android
Samsung


### Two rules about `super()`

`super()` has limits — the cells below show them (both error on purpose).

In [ ]:
# Rule 1: super() cannot be used OUTSIDE the class
class Phone:
    def __init__(self, price, brand, camera):
        print("Inside phone constructor")
        self.brand = brand

class Smartphone(Phone):
    def buy(self):
        print("Buying a smartphone")

s = Smartphone(20000, "Samsung", 12)
s.super().buy   # ❌ super() only works INSIDE a class, not out here

Inside phone constructor


In [ ]:
# Rule 2: super() cannot access parent ATTRIBUTES, only methods
class Phone:
    def __init__(self, price, brand, camera):
        print("Inside phone constructor")
        self.brand = brand
    def buy(self):
        print("buying a phone")

class Smartphone(Phone):
    def buy(self):
        print("Buying a smartphone")
        print(super().brand)   # ❌ super() can't reach attributes

s = Smartphone(20000, "Samsung", 12)
s.buy()

Inside phone constructor
Buying a smartphone


> 📌 **Remember about `super()`:**
> 1. It **cannot** access variables (attributes) — only methods/constructor.
> 2. It **cannot** be used outside a class.
> 3. It is **always** used inside a class.

### 🏅 Summary of Inheritance

1. A class can inherit from another class.
2. Inheritance improves **code reuse**.
3. The **constructor, attributes, and methods** get inherited.
4. The parent has **no access** to the child, but the child **can access** the parent.
5. **Private** properties of the parent are **not** directly accessible in the child.
6. A child can **override** the parent's attributes/methods (method overriding).
7. `super()` calls the parent's methods and constructor (not its variables).

### More `super()` examples

In [ ]:
# super() passing a value to the parent constructor
class Parent:
    def __init__(self, num):
        self.__num = num
    def get_num(self):
        return self.__num

class Child(Parent):
    def __init__(self, num, val):
        super().__init__(num)     # send num up to Parent's constructor
        self.__val = val
    def get_val(self):
        return self.__val

son = Child(100, 200)
print(son.get_num())   # 100 (from parent, via super)
print(son.get_val())   # 200 (from child)

100
200


In [ ]:
# self and the object are the same, so super()'s parent vars are visible
class Parent:
    def __init__(self):
        self.num = 100

class Child(Parent):
    def __init__(self):
        super().__init__()
        self.var = 200
    def show(self):
        print(self.num)   # from parent
        print(self.var)   # from child

son = Child()
son.show()

100
200


## 6️⃣ Types of Inheritance

There are five types:

| Type | Simple idea |
|------|-------------|
| **Single** | 1 parent → 1 child |
| **Multilevel** | grandparent → parent → child |
| **Hierarchical** | 1 parent → many children |
| **Multiple** | many parents → 1 child |
| **Hybrid** | a mix of the above |

### ➊ Single Inheritance — 1 parent, 1 child

```
   Phone   (parent)
     ▲
     │
 Smartphone (child)
```

In [ ]:
# single inheritance
class Phone:
    def __init__(self, price, brand, camera):
        print("Inside phone constructor")
        self.__price = price
        self.brand = brand
        self.camera = camera
    def buy(self):
        print("Buying a phone")

class Smartphone(Phone):
    pass

Smartphone(20000, "Samsung", "13px").buy()

Inside phone constructor
Buying a phone


### ➋ Multilevel Inheritance — grandparent → parent → child

```
  Product     (grandparent)
     ▲
  Phone       (parent)
     ▲
  Smartphone  (child)
```
The child can reach methods of *both* the parent and the grandparent.

In [ ]:
# multilevel inheritance
class Product:                 # grandparent
    def review(self):
        print("Product customer review")

class Phone(Product):          # parent (inherits Product)
    def __init__(self, price, brand, camera):
        print("Inside phone constructor")
        self.__price = price
        self.brand = brand
        self.camera = camera
    def buy(self):
        print("Buying a phone")

class Smartphone(Phone):       # child (inherits Phone)
    pass

s = Smartphone(20000, "Samsung", 12)
s.buy()      # from the parent (Phone)
s.review()   # from the grandparent (Product)

Inside phone constructor
Buying a phone
Product customer review


### ➌ Hierarchical Inheritance — 1 parent, many children

```
        Phone   (parent)
        ╱     ╲
 Smartphone   Featurephone   (children)
```

In [ ]:
# hierarchical inheritance — one parent, two children
class Phone:
    def __init__(self, price, brand, camera):
        print("Inside phone constructor")
        self.__price = price
        self.brand = brand
        self.camera = camera
    def buy(self):
        print("Buying a phone")

class Smartphone(Phone):
    pass

class Featurephone(Phone):
    pass

Smartphone(2000, "Samsung", "13px").buy()
Featurephone(1000, "lava", "1px").buy()

Inside phone constructor
Buying a phone
Inside phone constructor
Buying a phone


### ➍ Multiple Inheritance — many parents, 1 child

```
  Phone    Product   (two parents)
     ╲       ╱
     Smartphone      (one child)
```
The child inherits from **both** parents, so it can use methods from each.

In [ ]:
# multiple inheritance — one child, two parents
class Phone:
    def __init__(self, price, brand, camera):
        print("Inside phone constructor")
        self.__price = price
        self.brand = brand
        self.camera = camera
    def buy(self):
        print("Buying a phone")

class Product:
    def review(self):
        print("Customer review")

class Smartphone(Phone, Product):   # inherits BOTH
    pass

s = Smartphone(2000, "Samsung", 12)
s.buy()      # from Phone
s.review()   # from Product

Inside phone constructor
Buying a phone
Customer review


## 7️⃣ The Diamond Problem & MRO

**Problem:** what if **both parents have a method with the same name**? Which one runs?

🧠 **Analogy:** your father says "become a doctor", your mother says "become an engineer". Whom do you obey?

**The answer:** Python uses **MRO (Method Resolution Order)** — it obeys the parent whose name is written **first** in the brackets.

In [ ]:
class Phone:
    def __init__(self, price, brand, camera):
        print("Inside phone constructor")
        self.__price = price
        self.brand = brand
        self.camera = camera
    def buy(self):
        print("Buying a phone")

class Product:
    def buy(self):
        print("Product buy method")

# Phone is written FIRST, so Phone.buy() wins
class Smartphone(Phone, Product):
    pass

s = Smartphone(2000, "Samsung", 12)
s.buy()   # runs Phone's buy(), because Phone is listed first

Inside phone constructor
Buying a phone


In [ ]:
# MRO practice: work out the answer, then run it
class A:
    def m1(self):
        return 20

class B(A):
    def m1(self):
        return 30
    def m2(self):
        return 40

class C(B):
    def m2(self):
        return 20

obj1 = A()
obj3 = C()
# obj1.m1() -> 20 (A)
# obj3.m1() -> 30 (inherited from B)
# obj3.m2() -> 20 (C overrides B's m2)
print(obj1.m1() + obj3.m1() + obj3.m2())   # 20 + 30 + 20 = 70

70


### ⚠️ A common recursion bug with inheritance

To call a parent's method, use **`super().m1()`**, NOT `self.m1()`. Using `self.m1()` inside `m1` calls itself forever (infinite recursion).

In [ ]:
class A:
    def m1(self):
        return 20

class B(A):
    def m1(self):
        val = super().m1() + 30   # ✅ super() -> A.m1() = 20, so 50
        return val

class C(B):
    def m1(self):
        val = self.m1() + 20      # ❌ BUG: self.m1() calls C.m1() again -> infinite loop
        return val

print(A().m1())   # 20
print(B().m1())   # 50  (20 from A + 30)
# print(C().m1()) # would crash with RecursionError
# FIX: in C, write super().m1() + 20 instead of self.m1() + 20

20
50


### ➎ Hybrid Inheritance

**Hybrid** is a **mix** of two or more of the above types (like a joint family tree). You combine single, multiple, hierarchical, and multilevel as your application needs.

```
          A
         ╱ ╲
        B   C        (hierarchical from A)
         ╲ ╱
          D          (multiple from B and C)
```

---

# 🅲 Part 3 — Polymorphism

## What is Polymorphism?

**Polymorphism** means **"many faces"** — the same thing behaving **differently** depending on the situation.

It has **three** main forms:
1. **Method Overriding** — same method name in parent and child; the child's runs (we saw this above).
2. **Method Overloading** — same method name, different behaviour based on the inputs.
3. **Operator Overloading** — the same operator behaving differently for different inputs.

### Method Overloading

**Method overloading** = one method name that behaves differently based on how many arguments you pass.

> ⚠️ **Important:** Python does **not** truly support method overloading. If you write two methods with the same name, the **second one simply replaces the first**. But we can *fake* it using **default arguments**.

In [ ]:
# This does NOT work as overloading — the 2nd area() just replaces the 1st.
class Shape:
    def area(self, radius):
        return 3.14 * radius * radius
    def area(self, l, b):        # this replaces the method above
        return l * b
# Only the two-argument area() survives. Python has no real overloading.

**The trick:** use a **default argument** so one method handles both cases.

In [ ]:
# Faking overloading with a default argument
class Shape:
    def area(self, a, b=0):
        if b == 0:
            return 3.14 * a * a   # one argument -> treat as a circle
        else:
            return a * b          # two arguments -> treat as a rectangle

s = Shape()
print(s.area(2))      # circle:    3.14 * 2 * 2 = 12.56
print(s.area(3, 4))   # rectangle: 3 * 4 = 12

12.56
12


### Operator Overloading

The **same operator** does different things depending on the data type. Look at `+`:

In [ ]:
"hello" + "world"   # on strings, + joins (concatenates)

'helloworld'

In [ ]:
4 + 5   # on numbers, + adds

9

In [ ]:
[1, 2, 3] + [4, 5, 6]   # on lists, + merges

[1, 2, 3, 4, 5, 6]

Same operator `+`, three different behaviours — that is **operator overloading**. (Remember how we gave `+` a custom meaning for our `Fraction` class using `__add__`? That was operator overloading too!)

---

# 🅳 Part 4 — Abstraction

## What is Abstraction?

**Abstraction** means **hiding the complex details** and showing only what is necessary.

🧠 **Real-life examples:**
- A **laptop** has millions of complex parts inside, but you only need the surface — the keyboard and mouse. *How* it works is hidden.
- **Electromagnetic waves** are invisible (hidden), yet you use them to talk on the phone and use the internet.

### Abstraction in software

Imagine you are a **senior developer** building a bank's core application. The junior developers who build the web and mobile apps must follow a **rule**: *"If you want to inherit my class, you MUST also write a security function"* — because banking apps must be secure.

So abstraction lets a senior developer **force constraints** on the classes below them.

**Two key terms:**
- **Abstract class** — a class with **at least one abstract method**. (It can also have normal methods.)
- **Abstract method** — a method with **no code inside** (no implementation). The child class is *forced* to fill it in.

> 📌 **You cannot create an object of an abstract class** — it is incomplete.

### Example — a `Shape` abstract class

Every shape has an `area()` and `perimeter()`, but *how* they're calculated depends on the shape. So we make them **abstract** and let each shape fill in the details. We use Python's `abc` module (Abstract Base Class).

In [ ]:
# An abstract base class (an incomplete class)
from abc import ABC, abstractmethod

class Shape(ABC):              # ABC = Abstract Base Class
    @abstractmethod
    def area(self):
        pass

    @abstractmethod
    def perimeter(self):
        pass
# "Every shape MUST have an area and a perimeter — but I won't say how."

In [ ]:
# A concrete (complete) class fills in the formulas
class Rectangle(Shape):
    def __init__(self, length, width):
        self.length = length
        self.width = width
    def area(self):
        return self.length * self.width
    def perimeter(self):
        return 2 * (self.length + self.width)
# Valid, because it defines BOTH abstract methods.

In [ ]:
# Putting it all together
from abc import ABC, abstractmethod

class Shape(ABC):
    @abstractmethod
    def area(self):
        pass
    @abstractmethod
    def perimeter(self):
        pass

class Rectangle(Shape):
    def __init__(self, length, width):
        self.length = length
        self.width = width
    def area(self):
        return self.length * self.width
    def perimeter(self):
        return 2 * (self.length + self.width)

r = Rectangle(10, 5)
print("Area:", r.area())            # 50
print("Perimeter:", r.perimeter())  # 30

Area: 50
Perimeter: 30


### The bank app example

`BankApp` is abstract: `database()` is a normal method, but `security()` is **abstract** — so any child MUST implement it. This is how the senior developer forces the rule.

In [ ]:
from abc import ABC, abstractmethod

class BankApp(ABC):                 # an abstract class
    def database(self):             # a normal (concrete) method
        print("connected to database")

    @abstractmethod                 # an abstract method (no code)
    def security(self):
        pass

In [ ]:
class MobileApp(BankApp):            # inherits BankApp
    def mobile_login(self):
        print("login into mobile")

    def security(self):             # FORCED to implement security()
        print("mobile security")

In [ ]:
mob = MobileApp()   # allowed, because MobileApp implemented security()

In [ ]:
mob.database()   # can use the inherited normal method

connected to database


In [ ]:
mob.security()   # its own implementation of the forced method

mobile security


> ⚠️ If `MobileApp` had **not** implemented `security()`, Python would refuse to let you create `MobileApp()` — that is the power of abstraction: it forces child classes to complete the important methods.

### Why use abstract methods?

- **Forces** every subclass to implement important methods.
- Helps **avoid incomplete code**.
- Makes the design **clean and consistent**.

---

## 🎯 Summary — Quick Revision

Huge milestone — you finished the core of OOP! 🎉

| Concept | Key idea |
|---------|----------|
| **Aggregation** | "has-a": one class owns another (Customer has an Address) |
| **Inheritance** | "is-a": a child reuses a parent's code (`class Child(Parent)`) |
| **What's inherited** | Constructor, non-private attributes, non-private methods |
| **Method overriding** | Same method name in child & parent → child's runs |
| **`super()`** | Call the parent's methods/constructor (not its variables) |
| **Single / Multilevel / Hierarchical / Multiple / Hybrid** | The 5 inheritance types |
| **Diamond problem / MRO** | With two same-named parent methods, the first-listed parent wins |
| **Polymorphism** | Same thing, many behaviours (override, overload, operator overload) |
| **Abstraction** | Hide details; force child classes to implement key methods via `@abstractmethod` |

---

### 📝 Try It Yourself (Practice Questions)

1. Create a `Vehicle` parent class and `Car` / `Bike` child classes that override a `sound()` method.
2. Use `super()` in a `Manager` class that inherits from `Employee` and adds a bonus.
3. Build an aggregation: a `Library` class that *has* a list of `Book` objects.
4. Make an abstract class `Animal` with an abstract `speak()` method, then create `Dog` and `Cat`.
5. Give a custom `Vector` class operator overloading so `v1 + v2` adds two vectors.

> 💪 **Remember:** OOP is how real, large software is built. Whenever you design an app, ask: *"What are my classes, and how do they relate — has-a or is-a?"*

---

**➡️ Next up: the OOP Project — where we put all of this together!**